In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
import numpy as np
import pickle

In [9]:
NUM_FOLDS = 5
NUM_CLASSES = 3
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

LMDB_ROOT = "./lmdbs"

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [11]:
import lmdb
import pickle
import io
from PIL import Image
from torch.utils.data import Dataset

class LMDBDataset(Dataset):
    def __init__(self, lmdb_path, transform=None):
        self.lmdb_path = lmdb_path
        self.transform = transform

        # Open once ONLY to read keys, then close
        env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            self.keys = [k for k, _ in txn.cursor() if k != b"__len__"]

        env.close()

        # 🔥 Critical: length derived from keys, not __len__
        self.length = len(self.keys)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError

        # Open LMDB locally (safe for Windows)
        env = lmdb.open(
            self.lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            data = pickle.loads(txn.get(self.keys[idx]))

        env.close()

        img = Image.open(io.BytesIO(data["image"])).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, data["label"]


In [12]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def validate(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


In [13]:
from torchvision.models import alexnet, AlexNet_Weights

fold_results = []

for fold in range(NUM_FOLDS):
    print(f"\n===== Fold {fold} =====")

    train_lmdb = f"{LMDB_ROOT}/fold_{fold}_train.lmdb"
    val_lmdb = f"{LMDB_ROOT}/fold_{fold}_val.lmdb"

    train_dataset = LMDBDataset(train_lmdb, transform=train_transform)
    val_dataset = LMDBDataset(val_lmdb, transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # Model
    weights = AlexNet_Weights.DEFAULT
    model = alexnet(weights=weights)
    
    model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
 
    scaler = torch.amp.GradScaler("cuda")

    best_val_acc = 0

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )

        val_loss, val_acc = validate(
            model, val_loader, criterion
        )

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                model.state_dict(),
                f"alexnet_fold_{fold}.pth"
            )

    fold_results.append(best_val_acc)


===== Fold 0 =====
Epoch 1/15


C:\Users\istva\AppData\Local\Temp\ipykernel_19396\247927021.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Train Loss: 0.0469 | Train Acc: 0.9822 | Val Loss: 3.7808 | Val Acc: 0.7566
Epoch 2/15
Train Loss: 0.0158 | Train Acc: 0.9944 | Val Loss: 6.0646 | Val Acc: 0.7404
Epoch 3/15
Train Loss: 0.0071 | Train Acc: 0.9975 | Val Loss: 7.0664 | Val Acc: 0.7333
Epoch 4/15
Train Loss: 0.0055 | Train Acc: 0.9982 | Val Loss: 7.5881 | Val Acc: 0.7350
Epoch 5/15
Train Loss: 0.0034 | Train Acc: 0.9989 | Val Loss: 8.6894 | Val Acc: 0.7464
Epoch 6/15
Train Loss: 0.0040 | Train Acc: 0.9987 | Val Loss: 4.0515 | Val Acc: 0.7289
Epoch 7/15
Train Loss: 0.0025 | Train Acc: 0.9992 | Val Loss: 6.3440 | Val Acc: 0.7440
Epoch 8/15
Train Loss: 0.0028 | Train Acc: 0.9992 | Val Loss: 8.4862 | Val Acc: 0.7287
Epoch 9/15
Train Loss: 0.0025 | Train Acc: 0.9992 | Val Loss: 8.7625 | Val Acc: 0.7355
Epoch 10/15
Train Loss: 0.0021 | Train Acc: 0.9994 | Val Loss: 7.5012 | Val Acc: 0.7363
Epoch 11/15
Train Loss: 0.0025 | Train Acc: 0.9992 | Val Loss: 6.4743 | Val Acc: 0.7372
Epoch 12/15
Train Loss: 0.0020 | Train Acc: 0.9994 |

In [22]:
IDX_TO_CLASS = {
    0: "Normal",
    1: "Pneumonia",
    2: "COVID-19"
}

In [23]:
test_dataset = LMDBDataset(
    lmdb_path="./lmdbs/test.lmdb",
    transform=val_transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,   # LMDB-safe
    pin_memory=True
)

In [25]:
from torchvision.models import alexnet
import torch.nn as nn

def load_fold_model(weight_path):
    model = alexnet(weights=None)
    model.classifier[6] = nn.Linear(4096, 3)
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model


models = [
    load_fold_model(f"alexnet_fold_{i}.pth")
    for i in range(5)
]

In [26]:
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits_sum = None

        for model in models:
            with torch.amp.autocast("cuda"):
                logits = model(images)

            if logits_sum is None:
                logits_sum = logits
            else:
                logits_sum += logits

        avg_logits = logits_sum / len(models)
        preds = avg_logits.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [27]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Test Accuracy:", accuracy_score(all_labels, all_preds))

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=[IDX_TO_CLASS[i] for i in range(3)]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.3740516416958878
              precision    recall  f1-score   support

      Normal       0.95      0.28      0.43     15968
   Pneumonia       0.56      0.01      0.02      7965
    COVID-19       0.27      0.97      0.42      7437

    accuracy                           0.37     31370
   macro avg       0.59      0.42      0.29     31370
weighted avg       0.69      0.37      0.32     31370

Confusion Matrix:
[[ 4435     9 11524]
 [   93    83  7789]
 [  164    57  7216]]


In [28]:
import pandas as pd

df = pd.DataFrame({
    "gt_label": [IDX_TO_CLASS[i] for i in all_labels],
    "pred_label": [IDX_TO_CLASS[i] for i in all_preds]
})

df.to_csv("test_predictions_ensemble.csv", index=False)

In [29]:
for i in range(10):
    print(
        "GT:", IDX_TO_CLASS[all_labels[i]],
        "PRED:", IDX_TO_CLASS[all_preds[i]]
    )


GT: Pneumonia PRED: COVID-19
GT: Pneumonia PRED: COVID-19
GT: Pneumonia PRED: COVID-19
GT: Pneumonia PRED: COVID-19
GT: Pneumonia PRED: COVID-19
GT: COVID-19 PRED: COVID-19
GT: COVID-19 PRED: COVID-19
GT: COVID-19 PRED: COVID-19
GT: COVID-19 PRED: COVID-19
GT: COVID-19 PRED: COVID-19


In [46]:
from collections import Counter

def label_distribution(lmdb_path):
    ds = LMDBDataset(lmdb_path)
    labels = [ds[i][1] for i in range(len(ds))]
    return Counter(labels)

print("TRAIN:", label_distribution(train_lmdb))
print("VAL:",   label_distribution(val_lmdb))


TRAIN: Counter({0: 1157})
VAL: Counter({2: 15454, 1: 1657, 0: 1486})


In [1]:
import h5py
import torch
torch.backends.cudnn.benchmark = True
from torch.utils.data import Dataset
import numpy as np
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.nn as nn

class HDF5KFoldDataset(Dataset):
    def __init__(self, h5_path, fold, split, transform=None):
        self.h5_path = h5_path
        self.fold = str(fold)
        self.split = split
        self.transform = transform

        self.data = []
        self.labels = []

        self.class_map = {
            "Normal": 0,
            "COVID-19": 1,
            "Pneumonia": 2
        }

        with h5py.File(self.h5_path, 'r') as f:
            base = f[f"folds/{self.fold}/{self.split}"]

            for cls_name, label in self.class_map.items():
                dataset = base[cls_name]   # <-- this is a Dataset
                num_samples = dataset.shape[0]

                for i in range(num_samples):
                    self.data.append((cls_name, i))
                    self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        cls_name, i = self.data[idx]
        label = self.labels[idx]

        with h5py.File(self.h5_path, 'r') as f:
            img = f[f"folds/{self.fold}/{self.split}/{cls_name}"][i]

        # Convert to float32 and scale
        img = img.astype("float32") / 255.0

        # Handle grayscale vs RGB
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=0)   # (3, H, W)
        elif img.ndim == 3 and img.shape[-1] == 3:
            img = img.transpose(2, 0, 1)        # (3, H, W)

        img = torch.from_numpy(img)

        if self.transform:
            img = self.transform(img)

        return img, label

In [2]:
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [5]:
dataset = HDF5KFoldDataset("alexnet_image_data.h5", 0, "train")

print(len(dataset))

for i in range(5):
    img, label = dataset[i]
    print(i, img.shape, label)


302932
0 torch.Size([3, 227, 227]) 0
1 torch.Size([3, 227, 227]) 0
2 torch.Size([3, 227, 227]) 0
3 torch.Size([3, 227, 227]) 0
4 torch.Size([3, 227, 227]) 0


In [9]:
num_epochs = 10
num_folds = 5
h5_file = 'alexnet_image_data.h5'

for fold in range(num_folds):
    print(f"\n===== Fold {fold} =====")
    model = models.alexnet(weights=True)
    model.classifier[6] = nn.Linear(4096, 3)
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_dataset = HDF5KFoldDataset(h5_file, fold, "train", transform)
    val_dataset   = HDF5KFoldDataset(h5_file, fold, "val", transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_dataset, batch_size=32, num_workers=0, pin_memory=False)

    print("Dataset length:", len(train_dataset))
    img, label = train_dataset[0]
    print("Sample OK")

    for epoch in range(num_epochs):
        loss = train_one_epoch(model, train_loader)
        print("train done")
        acc = validate(model, val_loader)
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, Val Acc={acc:.4f}")


===== Fold 0 =====
Dataset length: 302932
Sample OK


KeyboardInterrupt: 